# Light Pollution x Air Quality Project

Resources:
- Google Earth Engine (GEE)
- light pollution dataset: [VIIRS Stray Light Corrected Nighttime Day/Night Band Composites Version 1](https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG)
- first air pollution dataset: [Sentinel-5P OFFL NO2: Offline Nitrogen Dioxide](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_OFFL_L3_NO2)


### project setup

In [80]:
# standard library imports
from datetime import datetime, timedelta
from pprint import pprint
# 3rd party imports
import ee
from geemap.foliumap import Map as FoliumMap

In [81]:
# establish access to GEE
ee.Authenticate()
ee.Initialize(project='cybernetic-tide-309501')

# verify successful GEE authentication
print(ee.String('Hello from the Earth Engine servers!').getInfo())

Hello from the Earth Engine servers!


In [82]:
# define helper functions

def get_mosaic(image_collection, date):
    """
    Mosaic is where you take an ImageCollection and flatten it to a single image. 
    It uses the pixels from the "top" images first, then fills in any holes with lower images
    We mosaic together data over a period of time so that we can fill in any holes from cloud cover

    returns: ee.image.Image
    """
    return (
        image_collection
        # Filter to a 2-month date rage so that our mosaic merges together that range of data
        .filterDate(date, date + timedelta(days=60))
        .mosaic()
    )

def date_is_in_range(date):
    """ 
    Returns whether the date is valid (in range for all relevant datasets) 
    light pollution date range: 2014-01-01T00:00:00Z–2025-03-01T00:00:00Z
    air pollution date range: 2018-06-28T10:24:07Z–2025-10-21T22:29:50Z

    returns: Boolean
    """
    min_date = datetime(2018, 6, 28)
    max_date = datetime(2025, 3, 1)
    return (min_date <= date <= max_date)

### user input
👀 make changes to this cell!

In [83]:
# Set ROI: update these two lines coordinates defining a rectangle
top_left_coordinate = (38.136560, -120.622745)
bottom_right_coordinate = (37.629004, -120.168263)

# Select your start date. Currently the data will be averaged over a two-month range (this is a static value we can change later). 
# Try mid-summer dates (in the northern hemisphere) to avoid cloud cover
start_date = datetime(2024, 6, 1)

### rearrange inputs

In [84]:
# make sure date is in range for the datasets before moving on!
assert date_is_in_range(start_date)

# convert from (lat, long) coordinates, to (x,y) vertices
y_max, x_min = top_left_coordinate
y_min, x_max = bottom_right_coordinate

# define ROI polygon as a sequence of points, forming a rectangle
# start in the bottom left, move clockwise through the vertices, and return back to the starting point.
roi_polygon = ee.Geometry.Polygon([
    [x_min, y_min],
    [x_min, y_max],
    [x_max, y_max],
    [x_max, y_min],
    [x_min, y_min],
])
roi_bounds = (x_min, y_min, x_max, y_max)

### fetch and aggregate data
1. choose [image collection](https://developers.google.com/earth-engine/apidocs/ee-imagecollection), filter for band and ROI bounds
2. get [mosaic](https://developers.google.com/earth-engine/apidocs/ee-imagecollection-mosaic) of image collection, i.e. composite all the images in the collection using a mask

TODO: look into [qualityMosaic](https://developers.google.com/earth-engine/apidocs/ee-imagecollection-qualitymosaic) to handle clouds

In [85]:
# VIIRS Stray Light Corrected Nighttime Day/Night Band Composites Version 1
viirs_collection = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG') \
    .select('avg_rad') \
    .filterBounds(roi_polygon)

# Sentinel-5P OFFL NO2: Offline Nitrogen Dioxide
no2_collection = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2') \
    .select('tropospheric_NO2_column_number_density') \
    .filterBounds(roi_polygon)

In [86]:
# aggregate data

# get nighttime data
nighttime = get_mosaic(viirs_collection, start_date) 
# TODO: look into cf_cvg band to exclude cloudy data? see mosaic docs...

# get NO2 data
tropospheric_no2 = get_mosaic(no2_collection, start_date)
# TODO: look into cloud_fraction band to exclude cloudy data?

In [96]:
# define visualization parameters
styled_roi = ee.FeatureCollection(ee.Feature(roi_polygon)).style(color='FFFFFFFF', fillColor='00000000')

nighttime_vis_params = { 'min': 0.0, 'max': 60.0 } # avg radiance values can go higher but that is excessive... Vegas will just be saturated

no2_vis_params = {
  'min': 0,
  'max': 0.0002,
  'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

ee.image.Image

### generate map

In [88]:
roi_map = FoliumMap()
roi_map.zoom_to_bounds(roi_bounds)

# light pollution data
roi_map.add_layer(nighttime, nighttime_vis_params, name='Nighttime')

# air pollution data
roi_map.add_layer(tropospheric_no2, no2_vis_params, name='Tropospheric NO2 Column Density')


# temp -> testing small ROI
y_min, x_max = bottom_right_coordinate
x_min = x_max - 0.01
y_max = y_min + 0.01
roi_of_interest = ee.Geometry.Polygon([ # TODO: decide on this pixel size
    [x_min, y_min],
    [x_min, y_max],
    [x_max, y_max],
    [x_max, y_min],
    [x_min, y_min],
])
roi_map.add_layer(roi_of_interest, {'color':'white'}, name='mini-ROI')


# add ROI indicator and display the map
roi_map.add_layer(styled_roi, name='ROI')
roi_map

### analysis

given Image objects nighttime and tropospheric_no2:
- use [reduceRegion](https://developers.google.com/earth-engine/apidocs/ee-image-reduceregion) to apply a Reducer to all pixels in a specific region (returns a Dictionary)
- use [reduce](https://developers.google.com/earth-engine/apidocs/ee-image-reduce) to apply a Reducer to all bands of an image (called at each pixel to reduce the stack of band values) (returns an Image)

In [98]:
# extract values for a small ROI:
scale=500     # FIXME
maxpixels=1e9 # FIXME
pixel_value_nighttime = nighttime.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi_of_interest, scale=scale, maxPixels=maxpixels)
pixel_value_no2 = tropospheric_no2.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi_of_interest, scale=scale, maxPixels=maxpixels)

ee.imagecollection.ImageCollection

In [90]:
radiance_val = pixel_value_nighttime.get('avg_rad').getInfo()
no2_val = pixel_value_no2.get('tropospheric_NO2_column_number_density').getInfo()
print(f"sampled at location {y_min + 0.05}, {x_max -0.05}: \n  radiance = {radiance_val:{1}.{4}} nanoWatts/sr/cm^2 \n  NO2 column density = {no2_val:{1}.{3}} mol/m^2")

sampled at location 37.679004, -120.218263: 
  radiance = 0.5132 nanoWatts/sr/cm^2 
  NO2 column density = 5.63e-05 mol/m^2


In [91]:
# should be using ee.Reducer.mean(), not .first(), since that returns the value from the first image in the collection... 
#  ... except this is run on an image already so they should be equal??
nighttime_val = nighttime.reduceRegion(reducer=ee.Reducer.autoHistogram(maxBuckets=6), geometry=roi_of_interest, scale=scale, maxPixels=maxpixels)
val = nighttime_val.get('avg_rad').getInfo()
print(val)

[[0.4375, 0.6941176470588235], [0.46875, 1.196078431372549], [0.5, 1.1372549019607843], [0.53125, 1.1137254901960785], [0.5625, 0], [0.59375, 0.8588235294117647]]


In [92]:
test_map = FoliumMap()
test_map.zoom_to_bounds(roi_bounds)
test_map.add_layer(nighttime, nighttime_vis_params, name='nighttime')
test = nighttime.reduce(reducer=ee.Reducer.stdDev())
test_map.add_layer(test, nighttime_vis_params, name='test')
test_map.add_layer(styled_roi, name='ROI')
test_map

In [114]:
# get data for two different times for comparison
light2024 = get_mosaic(viirs_collection, datetime(2024, 6, 1))
air2024 = get_mosaic(no2_collection, datetime(2024, 6, 1))
light2021 = get_mosaic(viirs_collection, datetime(2021, 6, 1))
air2021 = get_mosaic(no2_collection, datetime(2021, 6, 1))

# generate comparison images
light_air_comparison2024 = light2024.subtract(air2024.multiply(10000)) # just an example with a randomly chosen comparison
light_change = light2024.subtract(light2021)
air_change = air2024.subtract(air2021)

test_map2 = FoliumMap()
test_map2.zoom_to_bounds(roi_bounds)
test_map2.add_layer(light_air_comparison2024, {'palette': ['purple','blue','green','yellow','orange','red']}, name='light pollution vs no2 2024')
test_map2.add_layer(light_change, {'palette': ['purple','blue','green','yellow','orange','red']}, name='light pollution change from 2021-2024')
test_map2.add_layer(air_change, {'palette': ['purple','blue','green','yellow','orange','red']}, name='air pollution change from 2021-2024')
test_map2